# 📦 데이터 검증 및 번들 릴리스

이 노트북은 생성된 합성 데이터를 종합적으로 검증하고,
학습자용 **준비된 데이터 번들**로 패키징합니다.

## 검증 항목

| # | 검증 | 설명 |
|---|------|------|
| 1 | 종합 검증 | 스키마, 근거, 조건, 도구 호출 일관성 |
| 2 | 분할 누출 | 시나리오 패밀리 격리, 평가 오염 검사 |
| 3 | 백엔드 변환 | LoRA/OSFT 전용 포맷 내보내기 |
| 4 | 번들 빌드 | 매니페스트, 체크섬, 메타데이터 |
| 5 | 로더 검증 | 양쪽 백엔드의 로더/토크나이저 확인 |
| 6 | 데이터셋 카드 | 문서화 및 발행 |

## 번들 출력

```
tau-knowledge-v1/
├── manifest.json, checksums.sha256
├── canonical/{train,validation}.jsonl
├── training/{lora,osft}/{train,validation}.jsonl
├── kb/documents.jsonl
├── metadata/{provenance.jsonl, splits.json}
├── reports/{quality.json, quality.md, backend-validation.json}
├── DATASET_CARD.md, LICENSES/
└── configs/{lora.yaml, osft.yaml}
```

In [ ]:
"""Run comprehensive validation."""

import os
import sys
import json
import subprocess
from pathlib import Path
from collections import Counter

from rhoai_model_training_lab.config import (
    load_env, load_yaml_config, load_bundle_config, PROJECT_ROOT,
)

load_env()

sdg_config = load_yaml_config("configs/sdg.yaml")
prep_config = load_yaml_config("configs/data-preparation.yaml")
release_config = load_bundle_config()

canonical_path = PROJECT_ROOT / sdg_config["output"]["canonical_path"]

print("=" * 70)
print("🔍 종합 데이터 검증")
print("=" * 70)

validation_results = {
    "schema_valid": 0,
    "schema_invalid": 0,
    "grounding_ok": 0,
    "tool_valid": 0,
    "type_counts": Counter(),
    "errors": [],
}

if canonical_path.exists():
    from rhoai_model_training_lab.schemas.data import CanonicalSample

    all_samples = []
    for cf in sorted(canonical_path.glob("*.jsonl")):
        with open(cf) as f:
            for lineno, line in enumerate(f, 1):
                if not line.strip():
                    continue
                try:
                    data = json.loads(line)
                    sample = CanonicalSample(**data)
                    all_samples.append(sample)
                    validation_results["schema_valid"] += 1
                    validation_results["type_counts"][sample.sample_type.value] += 1

                    # Check tool schema consistency
                    if sample.tools:
                        validation_results["tool_valid"] += 1

                    # Check grounding
                    if sample.source_doc_ids:
                        validation_results["grounding_ok"] += 1

                except Exception as exc:
                    validation_results["schema_invalid"] += 1
                    validation_results["errors"].append(f"{cf.name}:{lineno}: {exc}")

    total = validation_results["schema_valid"] + validation_results["schema_invalid"]
    print(f"  총 레코드: {total}")
    print(f"  스키마 유효: {validation_results['schema_valid']}")
    print(f"  스키마 무효: {validation_results['schema_invalid']}")
    print(f"  도구 포함: {validation_results['tool_valid']}")
    print(f"  근거 문서 포함: {validation_results['grounding_ok']}")

    print("\n  유형별 분포:")
    for stype, count in validation_results["type_counts"].most_common():
        pct = count / max(total, 1) * 100
        print(f"    {stype:<25} {count:>5} ({pct:.1f}%)")

    if validation_results["errors"]:
        print(f"\n  오류 ({len(validation_results['errors'])}건):")
        for err in validation_results["errors"][:10]:
            print(f"    ❌ {err}")
else:
    print(f"  ❌ 정규 데이터 경로 없음: {canonical_path}")
    print("     02_generate_synthetic.ipynb를 먼저 실행하세요.")
    all_samples = []

In [ ]:
"""Check split leakage and contamination."""

print("=" * 70)
print("🔒 분할 누출 및 오염 검사")
print("=" * 70)

if all_samples:
    split_config = prep_config["splits"]

    # Group by scenario family
    family_groups = {}
    for sample in all_samples:
        family = sample.scenario_family or "unknown"
        if family not in family_groups:
            family_groups[family] = []
        family_groups[family].append(sample.sample_id)

    print(f"  시나리오 패밀리 수: {len(family_groups)}")
    print(f"  분할 방법: {split_config['method']}")
    print(f"  패밀리 격리: {split_config['family_isolation']}")

    # Check for potential leakage
    print("\n  패밀리 크기 분포:")
    sizes = [len(ids) for ids in family_groups.values()]
    print(f"    평균: {sum(sizes)/len(sizes):.1f}")
    print(f"    최소/최대: {min(sizes)} / {max(sizes)}")

    # Check for duplicate sample IDs
    all_ids = [s.sample_id for s in all_samples]
    unique_ids = set(all_ids)
    duplicates = len(all_ids) - len(unique_ids)
    if duplicates > 0:
        print(f"\n  ❌ 중복 ID 발견: {duplicates}건")
    else:
        print(f"\n  ✅ 중복 ID 없음")

    # Near-duplicate check (simplified)
    print("\n  중복 검사:")
    content_hashes = set()
    near_dupes = 0
    for sample in all_samples:
        content = " ".join(m.content or "" for m in sample.messages if m.content)
        import hashlib
        h = hashlib.md5(content.encode()).hexdigest()
        if h in content_hashes:
            near_dupes += 1
        content_hashes.add(h)

    print(f"    정확 중복: {near_dupes}건")
    print(f"    {'✅' if near_dupes == 0 else '⚠️ '} {'중복 없음' if near_dupes == 0 else '중복 제거 필요'}")

    # Contamination check note
    print("\n  오염 검사:")
    print("    ℹ️  평가 태스크와의 오염 검사는 격리된 프로세스에서 실행해야 합니다.")
    print("    최소한의 pass/fail 및 해시 기반 결과만 반환합니다.")
    print("    수동 실행: python scripts/validate_synthetic.py --contamination-check")
else:
    print("  ⚠️  검사할 데이터가 없습니다.")

In [ ]:
"""Export to backend-specific formats."""

print("=" * 70)
print("📤 백엔드별 포맷 내보내기")
print("=" * 70)

bundle_base = PROJECT_ROOT / release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")

if all_samples:
    # Split samples
    train_ratio = prep_config["splits"]["train_ratio"]
    seed = prep_config["splits"]["seed"]

    import random
    rng = random.Random(seed)

    # Group by family and assign splits
    families = list(family_groups.keys())
    rng.shuffle(families)

    split_point = int(len(families) * train_ratio)
    train_families = set(families[:split_point])
    val_families = set(families[split_point:])

    train_samples = [s for s in all_samples if (s.scenario_family or "unknown") in train_families]
    val_samples = [s for s in all_samples if (s.scenario_family or "unknown") in val_families]

    print(f"  학습 패밀리: {len(train_families)}, 검증 패밀리: {len(val_families)}")
    print(f"  학습 샘플: {len(train_samples)}, 검증 샘플: {len(val_samples)}")

    # Export canonical
    for split_name, samples in [("train", train_samples), ("validation", val_samples)]:
        out_dir = bundle_base / "canonical"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / f"{split_name}.jsonl"
        with open(out_file, "w") as f:
            for s in samples:
                f.write(s.model_dump_json() + "\n")
        print(f"  ✅ {out_file}: {len(samples)} 샘플")

    # Export backend-specific (LoRA and OSFT)
    for backend in ["lora", "osft"]:
        for split_name, samples in [("train", train_samples), ("validation", val_samples)]:
            out_dir = bundle_base / "training" / backend
            out_dir.mkdir(parents=True, exist_ok=True)
            out_file = out_dir / f"{split_name}.jsonl"

            with open(out_file, "w") as f:
                for s in samples:
                    # Convert to backend format (messages only, no provenance)
                    record = {
                        "messages": [m.model_dump(exclude_none=True) for m in s.messages],
                    }
                    if s.tools:
                        record["tools"] = [t.model_dump() for t in s.tools]
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
            print(f"  ✅ {out_file}: {len(samples)} 샘플")

    print("\n  ℹ️  LoRA/OSFT 데이터는 동일한 정규 샘플에서 변환됩니다.")
    print("     손실 마스킹: assistant 응답만 학습 대상")
else:
    print("  ⚠️  내보낼 데이터가 없습니다.")

In [ ]:
"""Build bundle with manifests and checksums."""

from rhoai_model_training_lab.data import compute_file_checksum
from rhoai_model_training_lab.schemas.data import BundleManifest

print("=" * 70)
print("📦 번들 빌드")
print("=" * 70)

if all_samples:
    model_id = release_config["model_profile"]["model_id"]
    model_revision = release_config["model_profile"]["model_revision"]

    # Build manifest
    type_dist = dict(validation_results["type_counts"])

    manifest = BundleManifest(
        bundle_name=release_config["bundle"]["name"],
        bundle_version=release_config["bundle"]["version"],
        tau_version=os.environ.get("TAU_BENCH_VERSION", ""),
        tau_commit_sha=os.environ.get("TAU_BENCH_COMMIT_SHA", ""),
        model_id=model_id,
        model_revision=model_revision,
        tokenizer_id=release_config["model_profile"]["tokenizer_id"],
        canonical_train_count=len(train_samples),
        canonical_validation_count=len(val_samples),
        split_policy=prep_config["splits"]["method"],
        sample_type_distribution=type_dist,
    )

    # Save manifest
    manifest_path = bundle_base / "manifest.json"
    with open(manifest_path, "w") as f:
        f.write(manifest.model_dump_json(indent=2))
    print(f"  ✅ 매니페스트: {manifest_path}")

    # Compute and save checksums
    checksum_path = bundle_base / "checksums.sha256"
    with open(checksum_path, "w") as f:
        for file_path in sorted(bundle_base.rglob("*")):
            if file_path.is_file() and file_path.name != "checksums.sha256":
                rel = file_path.relative_to(bundle_base)
                h = compute_file_checksum(file_path)
                f.write(f"{h}  {rel}\n")
    print(f"  ✅ 체크섬: {checksum_path}")

    # Save split metadata
    from rhoai_model_training_lab.schemas.data import SplitInfo
    split_info = SplitInfo(
        method=prep_config["splits"]["method"],
        seed=prep_config["splits"]["seed"],
        train_ids=[s.sample_id for s in train_samples],
        validation_ids=[s.sample_id for s in val_samples],
        train_count=len(train_samples),
        validation_count=len(val_samples),
        family_partition={
            f: "train" if f in train_families else "validation"
            for f in families
        },
    )
    meta_dir = bundle_base / "metadata"
    meta_dir.mkdir(parents=True, exist_ok=True)
    with open(meta_dir / "splits.json", "w") as f:
        f.write(split_info.model_dump_json(indent=2))
    print(f"  ✅ 분할 메타데이터: {meta_dir / 'splits.json'}")

    print(f"\n  번들 경로: {bundle_base}")
else:
    print("  ⚠️  빌드할 데이터가 없습니다.")

In [ ]:
"""Validate with both backend loaders."""

from rhoai_model_training_lab.data import BundleManager, validate_prepared_dataset

print("=" * 70)
print("🔍 백엔드 로더 검증")
print("=" * 70)

if bundle_base.exists() and (bundle_base / "manifest.json").exists():
    # Full validation
    validation = validate_prepared_dataset(bundle_base)

    if validation["valid"]:
        print("✅ 번들 검증 통과")
    else:
        print("❌ 번들 검증 실패")
        for err in validation["errors"]:
            print(f"  ❌ {err}")

    if validation["warnings"]:
        for warn in validation["warnings"]:
            print(f"  ⚠️  {warn}")

    # Test loading with both backends
    mgr = BundleManager.load_bundle(bundle_base)

    for backend in ["lora", "osft"]:
        print(f"\n  --- {backend.upper()} 로더 테스트 ---")
        try:
            train = mgr.get_training_samples(backend, "train")
            val = mgr.get_training_samples(backend, "validation")
            print(f"    학습: {len(train)} 샘플 ✅")
            print(f"    검증: {len(val)} 샘플 ✅")

            # Verify message format
            if train:
                first = train[0]
                assert "messages" in first, "messages 필드 누락"
                assert len(first["messages"]) > 0, "빈 메시지 목록"
                print(f"    메시지 포맷 ✅")
        except Exception as exc:
            print(f"    ❌ 실패: {exc}")

    # Tokenizer check
    model_id = release_config["model_profile"]["model_id"]
    compat = mgr.validate_compatibility(model_id)
    print(f"\n  모델 호환성: {'✅' if not compat.errors else '❌'}")
    for err in compat.errors:
        print(f"    ❌ {err}")
else:
    print("  ⚠️  번들이 존재하지 않습니다. 먼저 빌드하세요.")

In [ ]:
"""Generate dataset card and publish."""

print("=" * 70)
print("📄 데이터셋 카드 생성 및 발행")
print("=" * 70)

if bundle_base.exists() and (bundle_base / "manifest.json").exists():
    # Generate dataset card
    dataset_card = f"""# τ-Knowledge Banking Dataset Bundle

## Overview

- **Name**: {release_config['bundle']['name']}
- **Version**: {release_config['bundle']['version']}
- **Domain**: τ-Knowledge banking_knowledge
- **Model Target**: {release_config['model_profile']['model_id']}

## Contents

- Canonical training and validation samples
- LoRA and OSFT backend-specific exports
- KB document snapshot
- Provenance and split metadata
- Quality reports

## Generation Method

Synthetic data generated using sdg_hub with independent validation.
See `reports/quality.json` for detailed acceptance/rejection statistics.

## Validation Method

- Schema validation (Pydantic models)
- Grounding check against KB documents
- Tool schema consistency check
- Scenario family split isolation
- Evaluation contamination check (isolated process)

## Split Methodology

Scenario family-based splitting ensures that paraphrases,
name/number substitutions, and sibling examples remain in the same split.

## Limitations

- Synthetic data may not cover all banking scenarios
- Quality depends on teacher model capability
- This is a kb_adaptation experiment, not a training-free protocol
- Small model (4B) may not achieve high agent task performance

## Redistribution

Check LICENSES/ directory for applicable conditions.
"""

    card_path = bundle_base / "DATASET_CARD.md"
    with open(card_path, "w") as f:
        f.write(dataset_card)
    print(f"✅ 데이터셋 카드: {card_path}")

    # Create licenses directory
    licenses_dir = bundle_base / "LICENSES"
    licenses_dir.mkdir(exist_ok=True)
    print(f"✅ 라이선스 디렉토리: {licenses_dir}")

    # Publish options
    print("\n--- 발행 옵션 ---")
    publish_targets = release_config.get("publish", {}).get("targets", [])
    for target in publish_targets:
        target_type = target.get("type", "")
        if target_type == "s3":
            bucket = target.get("bucket", "")
            prefix = target.get("prefix", "")
            print(f"  📤 S3: s3://{bucket}/{prefix}")
            print(f"     aws s3 sync {bundle_base} s3://{bucket}/{prefix}{release_config['bundle']['name']}-{release_config['bundle']['version']}/")
        elif target_type == "local":
            local_path = target.get("path", "")
            print(f"  💾 로컬: {local_path}")

    # Build script reference
    print("\n  또는 빌드 스크립트 사용:")
    print(f"    python scripts/build_prepared_bundle.py --config configs/data-release.yaml")

    # Final summary
    print("\n" + "=" * 70)
    print("🎉 번들 준비 완료!")
    print("=" * 70)
    print(f"  번들 경로: {bundle_base}")
    if all_samples:
        print(f"  학습 샘플: {len(train_samples)}")
        print(f"  검증 샘플: {len(val_samples)}")
    print(f"  대상 모델: {release_config['model_profile']['model_id']}")
    print("\n  학습자는 이 번들을 다운로드하여 바로 학습을 시작할 수 있습니다.")
    print("  학습 노트북: notebooks/03_lora_finetuning.ipynb, 04_osft_finetuning.ipynb")
else:
    print("  ⚠️  번들이 존재하지 않습니다. 먼저 빌드하세요.")